# API exploration

This notebook runs a preflight check against both APIs before running the main pipeline to verify connectivity, credentials, and sensor availability.

### What this check verifies
* **Secret management (`dbutils.secrets.get`):** Confirms that the MET Frost client ID can be read securely from the `weather_price` secret scope without hardcoding credentials in the code.
* **Electricity price API (`Price status: 200`):** Fetches a sample day (`2025-01-15`) for `NO2` and confirms the response is a non-empty list of hourly records containing `NOK_per_kWh`, `EUR_per_kWh`, `EXR`, and local timestamps (`+01:00`).
* **Weather station metadata (`Source status: 200`):** Queries `/sources/v0.jsonld` to confirm that `SN44560` is the active Sola station in Rogaland (matching the `NO2` price area).
* **Available sensor time series (`Timeseries status: 200`):** Queries `/observations/availableTimeSeries/v0.jsonld` and verifies that `SN44560` has active hourly (`PT1H`) series (`validTo = None`) for `air_temperature`, `wind_speed`, and `sum(precipitation_amount PT1H)`. It also shows why we must filter for `PT1H` in Bronze, since Sola also reports 30-minute (`PT30M` with `PT20M` offset) aviation series.

In [0]:
FROST_CLIENT_ID = dbutils.secrets.get(scope="weather_price", key="frost_client_id")
print(FROST_CLIENT_ID)

In [0]:
import requests

# Read the MET Frost API key securely from Databricks Secrets (never hardcoded in Git)
FROST_CLIENT_ID = dbutils.secrets.get(
    scope="weather_price",
    key="frost_client_id"
)

# Verify electricity price API (NO2) on a sample date. Output will show whether we need to normalize to UTC in Silver
PRICE_URL = (
    "https://www.hvakosterstrommen.no/"
    "api/v1/prices/2025/01-15_NO2.json"
)
price_response = requests.get(PRICE_URL, timeout=20)
print("Price status:", price_response.status_code)
price_response.raise_for_status()
price_payload = price_response.json()
assert isinstance(price_payload, list) and len(price_payload) > 0
print(price_payload[:2])

# Verify Frost station metadata (SN44560 - Sola in Rogaland, matching price area NO2)
source_response = requests.get(
    "https://frost.met.no/sources/v0.jsonld",
    params={"ids": "SN44560", "types": "SensorSystem"},
    auth=(FROST_CLIENT_ID, ""),
    timeout=20
)
print("Source status:", source_response.status_code)
source_response.raise_for_status()
print(source_response.json().get("data", []))

In [0]:
# Verify available time series at Sola for temperature, wind speed, and precipitation
timeseries_response = requests.get(
    "https://frost.met.no/observations/availableTimeSeries/v0.jsonld",
    params={
        "sources": "SN44560",
        "elements": (
            "air_temperature,wind_speed,"
            "sum(precipitation_amount PT1H)"
        ),
        "referencetime": "2025-01-01/2025-03-31"
    },
    auth=(FROST_CLIENT_ID, ""),
    timeout=30
)
print("Timeseries status:", timeseries_response.status_code)
timeseries_response.raise_for_status()

available = timeseries_response.json().get("data", [])
for item in available:
    print(
        item.get("sourceId"),
        item.get("elementId"),
        item.get("timeResolution"),
        item.get("timeOffset"),
        item.get("validFrom"),
        item.get("validTo")
    )

In [0]:
# Fail fast if Sola does not have active hourly (PT1H) temperature and wind series
required = {"air_temperature", "wind_speed"}
required_hourly = {
    item.get("elementId")
    for item in available
    if item.get("timeResolution") == "PT1H"
}
missing_required = required - required_hourly
if missing_required:
    raise ValueError(
        f"SN44560 is missing required PT1H series: {missing_required}"
    )

print("Preflight passed for NO2 and SN44560.")